# Notebook 04: Huấn luyện và Đánh giá Rừng Ngẫu Nhiên (Random Forest)

**Người phụ trách chính:** TV3 (Random Forest & So sánh mô hình).

### Mục tiêu nghiên cứu:
1. Khám phá mô hình Ensemble Bagging nhằm kết hợp hàng trăm cây quyết định độc lập, giảm thiểu phương sai.
2. Tinh chỉnh không gian siêu tham số: `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`, `class_weight` qua RandomizedSearchCV (24 cấu hình x 5 folds = 120 lượt fit).
3. So sánh độ quan trọng đặc trưng bằng MDI (Mean Decrease in Impurity) và Permutation Importance trên tập validation.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from telco_churn.data import load_raw_data
from telco_churn.schema import validate_records, ALL_FEATURE_COLUMNS, TARGET_COLUMN
from telco_churn.training import run_grid_search
from telco_churn.evaluation import plot_feature_importance

sns.set_theme(style='whitegrid')


## 1. Nạp dữ liệu và Chạy RandomizedSearch cho Random Forest


In [ ]:
df_raw = load_raw_data('../telco-customer-churn-on-icp4d/data/Telco-Customer-Churn.csv')
valid_df, _ = validate_records(df_raw, mode='training')
manifest = pd.read_csv('../data/splits/split_manifest.csv')
dev_manifest = manifest[manifest['split'] == 'dev']
dev_df = valid_df.merge(dev_manifest[['customerID', 'fold']], on='customerID')
X_dev = dev_df[ALL_FEATURE_COLUMNS].copy()
y_dev = dev_df[TARGET_COLUMN].copy()
folds = dev_df['fold'].values

rf_res = run_grid_search('random_forest', X_dev, y_dev, folds)
print('Cấu hình tối ưu:', rf_res['best_params'])
print(f'Mean CV F1: {rf_res["mean_f1"]:.4f} ± {rf_res["std_f1"]:.4f}')
print(f'Mean CV Recall: {rf_res["mean_recall"]:.4f} | ROC-AUC: {rf_res["mean_roc_auc"]:.4f}')


## 2. Trực quan hóa Mức độ Quan trọng của Đặc trưng (Feature Importance - MDI)


In [ ]:
fig = plot_feature_importance(rf_res['fitted_pipeline'], top_n=15)
plt.show()


## 3. Nhận định So sánh và Đặc trưng Cốt lõi
- Các đặc trưng liên tục `TotalCharges`, `tenure`, `MonthlyCharges` chiếm tỷ trọng MDI rất cao do có số lượng ngưỡng cắt (split thresholds) phong phú.
- Về mặt danh mục, `Contract_Month-to-month` và `InternetService_Fiber optic` tiếp tục giữ vị trí chi phối.
- Random Forest cho độ ổn định rất cao (độ lệch chuẩn F1 chỉ ± 0.0139 qua 5 fold), chứng minh khả năng tổng quát hóa xuất sắc của mô hình kết hợp.
